<center><h1>DSCI 552 — Homework 3</h1></center>

Name: Wendu Liu
<br>
GitHub Username: liuwendu2004
<br>
USC ID: 8752809381

## 1. Time Series Classification Part 1: Feature Creation/Extraction

### (a) Download Data

Dataset: [UCI AReM (Activity Recognition system based on Multisensor data fusion)](https://archive.ics.uci.edu/dataset/366/activity+recognition+system+based+on+multisensor+data+fusion+arem).

The provided **`Homework 3, 4 Data.zip`** archive contains the `AReM/` dataset. Keep the archive in the same folder as this notebook. The code also accepts an extracted `AReM/` folder or a standalone `AReM.zip`. There are 88 activity recordings (instances); each recording contains six sensor time series, normally with 480 observations each. We do not use the time column as a seventh sensor.

**Package imports**

In [1]:
from pathlib import Path
import re
import zipfile

import numpy as np
import pandas as pd
from IPython.display import display

ACTIVITIES = ("bending1", "bending2", "cycling", "lying", "sitting", "standing", "walking")
SENSOR_COLUMNS = ("avg_rss12", "var_rss12", "avg_rss13", "var_rss13", "avg_rss23", "var_rss23")
EXPECTED_TIMES = pd.Index(np.arange(0, 120_000, 250), name="time")

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 50)

**Load and clean the AReM dataset**

In [2]:
def locate_dataset_files():
    """Return (activity, dataset_number, identifier, raw_csv_bytes) for all 88 files."""
    folders = [Path("AReM"), Path("data") / "AReM"]
    for folder in folders:
        if folder.is_dir():
            found = []
            for activity in ACTIVITIES:
                for path in (folder / activity).glob("dataset*.csv"):
                    match = re.fullmatch(r"dataset(\d+)\.csv", path.name)
                    if match:
                        found.append((activity, int(match.group(1)),
                                      f"{activity}/{path.name}", path.read_bytes()))
            if found:
                return sorted(found, key=lambda r: (ACTIVITIES.index(r[0]), r[1]))

    archives = [Path("Homework 3, 4 Data.zip"), Path("AReM.zip"),
                Path("data") / "Homework 3, 4 Data.zip"]
    pattern = re.compile(r"(?:AReM/)?(" + "|".join(ACTIVITIES) + r")/dataset(\d+)\.csv$")
    for archive in archives:
        if archive.is_file():
            with zipfile.ZipFile(archive) as zf:
                found = []
                for name in zf.namelist():
                    match = pattern.fullmatch(name)
                    if match:
                        activity, num = match.group(1), int(match.group(2))
                        found.append((activity, num, f"{activity}/dataset{num}.csv", zf.read(name)))
            if found:
                return sorted(found, key=lambda r: (ACTIVITIES.index(r[0]), r[1]))

    raise FileNotFoundError(
        "Place 'Homework 3, 4 Data.zip', 'AReM.zip', or an extracted 'AReM/' "
        "folder next to this notebook and run again."
    )


def parse_sensor_csv(raw_bytes, label):
    """Handle CSVs with whitespace delimiters, stray trailing commas, or missing rows."""
    observations = []
    for line in raw_bytes.decode("utf-8-sig").splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        fields = re.split(r"[,\s]+", line.rstrip(","))
        if len(fields) != 7:
            raise ValueError(f"Unexpected number of columns in {label}: {line!r}")
        observations.append([float(x) for x in fields])

    frame = pd.DataFrame(observations, columns=["time", *SENSOR_COLUMNS])
    frame["time"] = frame["time"].astype(int)
    if frame["time"].duplicated().any():
        raise ValueError(f"Duplicate timestamps in {label}")
    frame = frame.set_index("time").sort_index()
    if not frame.index.isin(EXPECTED_TIMES).all():
        raise ValueError(f"Unexpected timestamps in {label}")

    missing_rows = len(EXPECTED_TIMES.difference(frame.index))
    # One supplied file is missing timestamp 13500: interpolate from its neighbors.
    frame = frame.reindex(EXPECTED_TIMES).interpolate(method="index", limit_direction="both")
    if frame.isna().any().any() or len(frame) != 480:
        raise ValueError(f"Invalid or incomplete sensor readings in {label}")
    return frame, missing_rows

source_files = locate_dataset_files()
assert len(source_files) == 88, f"Expected 88 instances, found {len(source_files)}"

records = []
for activity, dataset_num, instance, raw in source_files:
    readings, repaired_rows = parse_sensor_csv(raw, instance)
    split = ("test" if dataset_num <= (2 if activity.startswith("bending") else 3)
             else "train")
    records.append({"instance": instance, "activity": activity,
                    "dataset": dataset_num, "split": split,
                    "interpolated_rows": repaired_rows, "readings": readings})

print(f"Loaded {len(records)} recordings, each with 6 series × 480 measurements.")
print("Missing time rows repaired by interpolation:", sum(r["interpolated_rows"] for r in records))
display(pd.DataFrame([{"activity": r["activity"], "split": r["split"]}
                      for r in records]).groupby(["activity", "split"]).size()
        .unstack(fill_value=0))

Loaded 88 recordings, each with 6 series × 480 measurements.
Missing time rows repaired by interpolation: 1


split,test,train
activity,,
bending1,2,5
bending2,2,4
cycling,3,12
lying,3,12
sitting,3,12
standing,3,12
walking,3,12


### (b) Test and Train Data

Reserve **datasets 1 and 2** from `bending1` and `bending2` as test instances. For the other five activities, reserve **datasets 1, 2, and 3** as test instances. All remaining instances belong to the training set. This produces 69 training recordings and 19 test recordings. The features in part (c) are extracted for **all 88** instances, as requested.

In [3]:
train_records = [r for r in records if r["split"] == "train"]
test_records = [r for r in records if r["split"] == "test"]

assert len(train_records) == 69
assert len(test_records) == 19
assert not ({r["instance"] for r in train_records} &
            {r["instance"] for r in test_records})

print(f"Training recordings: {len(train_records)}")
print(f"Test recordings:     {len(test_records)}")
print("Test instances by activity:")
display(pd.Series([r["activity"] for r in test_records]).value_counts().sort_index().to_frame("count"))

Training recordings: 69
Test recordings:     19
Test instances by activity:


,count
bending1,2
bending2,2
cycling,3
lying,3
sitting,3
standing,3
walking,3


### (c) Feature Extraction

#### i. Research

**Common time-domain features** for time-series classification include minimum, maximum, mean, median, standard deviation, variance, first quartile (25th percentile), third quartile (75th percentile), range, interquartile range, root mean square (RMS), skewness, kurtosis, mean absolute deviation, zero-crossing rate, and autocorrelation. These summarize central tendency, spread, shape, and temporal behavior without transforming the signal into the frequency domain.

For this assignment, we extract exactly the seven features specified in part (c)ii for each of the six sensor signals.

#### ii. Extraction

Compute **min, max, mean, median, sample standard deviation (ddof = 1), first quartile, and third quartile** for each of the six sensor series. This yields **7 × 6 = 42 features per recording**, organized into an 88 × 42 data table. The record's activity and train/test label are stored separately, not counted among the 42 predictors.

In [4]:
FEATURE_NAMES = ("min", "max", "mean", "median", "std", "1st_quart", "3rd_quart")


def extract_features(readings):
    features = {}
    for series_id, sensor in enumerate(SENSOR_COLUMNS, start=1):
        values = readings[sensor]
        features.update({
            f"min{series_id}": values.min(),
            f"max{series_id}": values.max(),
            f"mean{series_id}": values.mean(),
            f"median{series_id}": values.median(),
            f"std{series_id}": values.std(ddof=1),
            f"1st_quart{series_id}": values.quantile(0.25),
            f"3rd_quart{series_id}": values.quantile(0.75),
        })
    return features

feature_table = pd.DataFrame(
    [extract_features(r["readings"]) for r in records],
    index=pd.Index([r["instance"] for r in records], name="Instance"),
)
metadata = pd.DataFrame(
    [{"activity": r["activity"], "split": r["split"]} for r in records],
    index=feature_table.index,
)

assert feature_table.shape == (88, 42)
assert feature_table.notna().all().all()

X_train = feature_table.loc[metadata["split"] == "train"].copy()
X_test = feature_table.loc[metadata["split"] == "test"].copy()

print("Feature matrix shape (instances × extracted features):", feature_table.shape)
print("Training features:", X_train.shape, "  Test features:", X_test.shape)
display(feature_table.head(8).round(3))

# Keep the full 88 × 42 table for use in Homework 4.
feature_table.to_csv("HW3_AReM_42_features.csv")

Feature matrix shape (instances × extracted features): (88, 42)
Training features: (69, 42)   Test features: (19, 42)


,min1,max1,mean1,median1,std1,1st_quart1,3rd_quart1,min2,max2,mean2,median2,std2,1st_quart2,3rd_quart2,min3,max3,mean3,median3,std3,1st_quart3,3rd_quart3,min4,max4,mean4,median4,std4,1st_quart4,3rd_quart4,min5,max5,mean5,median5,std5,1st_quart5,3rd_quart5,min6,max6,mean6,median6,std6,1st_quart6,3rd_quart6
Instance,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
bending1/dataset1.csv,37.25,45.00,40.625,40.50,1.477,39.250,42.00,0.0,1.30,0.359,0.43,0.323,0.00,0.50,4.00,29.50,19.041,19.250,4.463,16.00,23.250,0.0,7.23,0.833,0.50,0.966,0.00,1.120,27.25,38.25,34.311,35.00,2.188,33.000,36.00,0.0,1.92,0.571,0.43,0.583,0.00,1.30
bending1/dataset2.csv,38.00,45.67,42.813,42.50,1.436,42.000,43.67,0.0,1.22,0.372,0.47,0.289,0.00,0.50,2.00,29.50,20.097,21.000,3.894,19.00,22.250,0.0,5.76,0.876,0.50,1.047,0.00,1.145,27.67,38.50,33.025,33.00,1.995,32.000,34.50,0.0,3.11,0.571,0.43,0.601,0.00,1.30
bending1/dataset3.csv,35.00,47.40,43.954,44.33,1.559,43.000,45.00,0.0,1.70,0.426,0.47,0.339,0.00,0.50,6.50,29.75,22.122,23.000,3.031,19.75,24.000,0.0,4.44,0.497,0.43,0.551,0.00,0.830,29.00,38.50,35.588,36.00,2.000,35.362,36.50,0.0,1.79,0.493,0.43,0.514,0.00,0.94
bending1/dataset4.csv,33.00,47.75,42.180,43.50,3.671,39.150,45.00,0.0,3.00,0.696,0.50,0.631,0.00,1.12,8.50,30.00,22.184,23.000,3.810,20.50,24.372,0.0,5.15,0.990,0.83,0.954,0.43,1.300,20.00,38.67,33.494,35.00,3.849,30.458,36.33,0.0,2.18,0.614,0.50,0.524,0.00,1.00
bending1/dataset5.csv,33.00,45.75,41.678,41.75,2.243,41.330,42.75,0.0,2.83,0.536,0.50,0.405,0.43,0.71,3.00,28.25,19.007,19.125,4.087,16.50,22.062,0.0,6.42,0.842,0.50,0.929,0.43,1.120,23.67,37.50,29.857,30.00,2.411,28.458,31.25,0.0,1.79,0.383,0.43,0.389,0.00,0.50
bending1/dataset6.csv,37.00,48.00,43.455,43.25,1.386,42.500,45.00,0.0,1.58,0.378,0.47,0.316,0.00,0.50,5.75,27.00,15.793,15.000,3.848,13.00,18.270,0.0,10.03,0.849,0.50,0.996,0.43,1.120,8.00,33.50,23.035,23.50,2.489,22.250,24.00,0.0,5.26,0.680,0.50,0.623,0.43,0.87
bending1/dataset7.csv,36.25,48.00,43.969,44.50,1.618,43.310,44.67,0.0,1.50,0.413,0.47,0.263,0.43,0.50,1.50,26.33,15.868,16.250,3.742,14.25,18.000,0.0,5.17,0.666,0.47,0.789,0.00,0.940,11.33,30.75,22.104,21.67,3.318,20.500,23.75,0.0,2.96,0.555,0.49,0.488,0.00,0.83
bending2/dataset1.csv,12.75,51.00,24.563,24.25,3.738,23.188,26.50,0.0,6.87,0.591,0.43,0.837,0.00,0.71,0.00,25.33,19.121,20.250,4.300,18.00,22.000,0.0,6.76,0.737,0.47,0.927,0.00,0.870,10.67,30.00,23.493,23.75,3.694,20.500,27.00,0.0,4.97,0.700,0.50,0.694,0.43,0.87


#### iii. Standard Deviation

Estimate the **sample standard deviation across the 88 instances** for each of the 42 extracted feature columns. A 90% percentile Bootstrap confidence interval is obtained by resampling the **88 recordings (rows)** with replacement 10,000 times, recalculating the sample standard deviation each time, and taking the 5th and 95th percentiles. Resampling whole rows preserves associations among the 42 features of an instance. A fixed seed ensures reproducibility.

In [5]:
values = feature_table.to_numpy(dtype=float)
feature_stds = values.std(axis=0, ddof=1)

rng = np.random.default_rng(552)
n_bootstrap = 10_000
bootstrap_stds = np.empty((n_bootstrap, values.shape[1]), dtype=float)

# Evaluate in batches to keep memory use small.
for start in range(0, n_bootstrap, 500):
    batch_size = min(500, n_bootstrap - start)
    draw = rng.integers(0, len(values), size=(batch_size, len(values)))
    bootstrap_stds[start:start + batch_size] = values[draw].std(axis=1, ddof=1)

ci_lower, ci_upper = np.percentile(bootstrap_stds, [5, 95], axis=0)
std_summary = pd.DataFrame({
    "standard_deviation": feature_stds,
    "90%_CI_lower": ci_lower,
    "90%_CI_upper": ci_upper,
}, index=pd.Index(feature_table.columns, name="Feature"))

assert std_summary.shape == (42, 3)
assert (std_summary["90%_CI_lower"] <= std_summary["90%_CI_upper"]).all()

display(std_summary.round(3))
std_summary.to_csv("HW3_feature_std_bootstrap_CI.csv")

,standard_deviation,90%_CI_lower,90%_CI_upper
Feature,,,
min1,9.570,8.262,10.761
max1,4.394,3.328,5.285
mean1,5.336,4.704,5.886
median1,5.440,4.790,6.004
std1,1.772,1.572,1.950
1st_quart1,6.154,5.582,6.652
3rd_quart1,5.139,4.332,5.841
min2,0.000,0.000,0.000
max2,5.063,4.623,5.402


#### iv. Select Features

I select **minimum, mean, and maximum** as the three most important time-domain features for this task.

- **Minimum** captures the lowest observed sensor reading in an activity and can help identify characteristic low positions or movements.
- **Mean** describes the overall sensor level across an activity; it is relatively stable because it uses all 480 measurements.
- **Maximum** captures peak values that may distinguish more intense movements from less active behaviors.

Together, these features describe the lower extreme, typical level, and upper extreme of each time series. They are straightforward to interpret and may help separate different activities. This is a judgment-based choice rather than a claim of proven predictive importance; one limitation is that minimum and maximum can be sensitive to outliers.

## 2. ISLR 3.7.4

### (a) Linear Train

The cubic regression is expected to have **training RSS less than or equal to** that of the linear regression. The cubic model contains the linear model as a special case when the coefficients of $X^2$ and $X^3$ are both zero. Least squares minimizes RSS over a larger set of possible fits, so adding these terms cannot increase training RSS, even if the true relationship is linear.

### (b) Linear Test

We would expect the **linear regression to have lower test RSS**. Since the true relationship is linear, the linear model is correctly specified. The extra cubic terms can fit random noise in the training data and increase prediction variance, making the cubic model more prone to overfitting. A cubic model could still do better on a particular test set by chance; the statement concerns expected performance.

### (c) Not Linear Train

The cubic regression has **training RSS less than or equal to** the linear regression's training RSS. The cubic model includes all possible linear fits, so its least-squares fit cannot have a larger training RSS, regardless of whether the true relationship is linear or nonlinear.

### (d) Not Linear Testing

There is **not enough information to determine** which model has lower test RSS. If the true relationship is strongly nonlinear and well approximated by a cubic polynomial, the cubic model may reduce bias and produce lower test error. If it is only slightly nonlinear, or its shape is not captured well by a cubic polynomial, the simpler linear model may generalize better due to its lower variance. The answer depends on the unknown relationship and the bias–variance trade-off.

## 3. ISLR 3.7.3 - Extra Practice 

*Extra practice only — not required for submission.*

## 3. ISLR 3.7.5 - Extra Practice 

*Extra practice only — not required for submission.*